# Phase 4 - Resolution Time Regression

Goal: predict `resolution_hours` from text + structured + temporal features. Target: MAE at least 10% better than the median-per-category baseline.

Phases 0-3 must be passing. Phase 2's `sample_2m_preprocessed.parquet` must be on Drive.

**Why log1p(target)?** Resolution times are heavy-tailed (some complaints take minutes, others months). A direct linear regression on raw hours overweights the tail and produces nonsensical negative predictions for fast tickets. We train on `log1p(hours)` and back-transform predictions for reporting.

## Cell 1 - Bootstrap

In [ ]:
REPO_URL = 'https://github.com/george-gideon-S/cs-gy-6513-big-data-311-nlp.git'

from google.colab import drive
drive.mount('/content/drive')

import subprocess, os, sys
if not os.path.isdir('/content/project/.git'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/project'], check=True)
else:
    subprocess.run(['git', '-C', '/content/project', 'pull'], check=True)

if '/content/project' not in sys.path:
    sys.path.insert(0, '/content/project')

!pip install -r /content/project/requirements.txt -q

!apt-get install -y openjdk-11-jre-headless > /dev/null 2>&1
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

import nltk
for pkg in ['stopwords', 'wordnet', 'punkt', 'punkt_tab', 'omw-1.4']:
    nltk.download(pkg, download_dir='/root/nltk_data', quiet=True)

from src.spark_setup import get_spark
spark = get_spark(app_name='phase4-regress')
print('spark', spark.version, 'ready')

## Cell 2 - Load + filter to closed complaints with valid resolution time

We need closed_date - created_date to be sane. Drop nulls, negatives, and absurdly long durations (anything > 1 year is usually a data quality issue). Keep only the top-20 categories so we evaluate against the same scope as Phase 3.

In [ ]:
from pyspark.sql import functions as F
from src.config import TOP_K_CATEGORIES

in_path = '/content/drive/MyDrive/cs6513/sample_2m_preprocessed.parquet'
df_full = spark.read.parquet(in_path)
print(f'loaded {df_full.count():,} rows')

# top-20 filter (same as phase 3)
top_classes = (
    df_full.groupBy('label_canonical').count()
    .orderBy(F.desc('count')).limit(TOP_K_CATEGORIES)
    .toPandas()['label_canonical'].tolist()
)

# compute resolution time. cap at 1 year - longer is almost always a data bug.
df = (
    df_full
    .filter(F.size('tokens') > 0)
    .filter(F.col('label_canonical').isin(top_classes))
    .filter(F.col('closed_date').isNotNull())
    .withColumn(
        'resolution_hours',
        (F.unix_timestamp('closed_date') - F.unix_timestamp('created_date')) / 3600.0
    )
    .filter(F.col('resolution_hours') > 0)
    .filter(F.col('resolution_hours') < 24 * 365)  # 1 year cap
)
n = df.count()
print(f'rows after filter: {n:,}')

df.select('resolution_hours').describe().show()

## Cell 3 - Add temporal features + log1p target

The log1p transform makes the target distribution roughly symmetric, which linear regression handles much better than the raw heavy-tailed distribution.

In [ ]:
df = (
    df
    .withColumn('hour_of_day', F.hour('created_date').cast('double'))
    .withColumn('day_of_week', F.dayofweek('created_date').cast('double'))
    .withColumn('log_resolution_hours', F.log1p('resolution_hours'))
)

# sanity check
df.select('resolution_hours', 'log_resolution_hours', 'hour_of_day', 'day_of_week').show(5)
df.select('log_resolution_hours').describe().show()

## Cell 4 - 80/20 split

For regression we dont need stratified splits the same way - we use random split.

In [ ]:
train, test = df.randomSplit([0.8, 0.2], seed=42)
n_train = train.count()
n_test = test.count()
print(f'train: {n_train:,}  test: {n_test:,}')

## Cell 5 - Build + fit pipeline

Pipeline: HashingTF + IDF (text) + StringIndexer/OneHot for agency + StringIndexer/OneHot for borough + assemble all + LinearRegression. Predicts log1p(hours).

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    HashingTF, IDF, StringIndexer, OneHotEncoder, VectorAssembler,
)
from pyspark.ml.regression import LinearRegression
import time

agency_idx = StringIndexer(inputCol='agency', outputCol='agency_idx', handleInvalid='keep')
borough_idx = StringIndexer(inputCol='borough', outputCol='borough_idx', handleInvalid='keep')
agency_oh = OneHotEncoder(inputCol='agency_idx', outputCol='agency_vec')
borough_oh = OneHotEncoder(inputCol='borough_idx', outputCol='borough_vec')

htf = HashingTF(inputCol='tokens', outputCol='raw_text', numFeatures=4096)
idf = IDF(inputCol='raw_text', outputCol='text_vec', minDocFreq=10)

assembler = VectorAssembler(
    inputCols=['text_vec', 'agency_vec', 'borough_vec', 'hour_of_day', 'day_of_week'],
    outputCol='features',
)

lr = LinearRegression(
    labelCol='log_resolution_hours',
    featuresCol='features',
    regParam=0.1,
    elasticNetParam=0.0,
    maxIter=50,
)

pipeline = Pipeline(stages=[
    agency_idx, borough_idx, agency_oh, borough_oh,
    htf, idf, assembler, lr,
])

t0 = time.time()
lr_model = pipeline.fit(train)
t_fit = time.time() - t0
print(f'fit in {t_fit:.1f} sec')

## Cell 6 - Evaluate (in original hours space, the metric we care about)

We back-transform predictions via expm1, then compute MAE / RMSE / R^2 in the original hours unit. Reporting in log space is misleading because the user-facing metric needs to be in real hours.

In [ ]:
from pyspark.sql import functions as F
import numpy as np

preds = lr_model.transform(test).select(
    'resolution_hours',
    F.expm1('prediction').alias('predicted_hours'),
)

# pull to pandas for clean numpy metrics
preds_pdf = preds.toPandas()
y_true = preds_pdf['resolution_hours'].values
y_pred = preds_pdf['predicted_hours'].values
y_pred = np.maximum(y_pred, 0.01)  # cant predict negative hours

mae = float(np.mean(np.abs(y_true - y_pred)))
rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
ss_res = float(np.sum((y_true - y_pred) ** 2))
ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

print('linear regression test metrics (in hours):')
print(f'  MAE  = {mae:.2f} hours  ({mae/24:.2f} days)')
print(f'  RMSE = {rmse:.2f} hours ({rmse/24:.2f} days)')
print(f'  R^2  = {r2:.4f}')

## Cell 7 - Median-per-category baseline

The model has to beat this by >=10% on MAE to be worth shipping. The baseline says "for any new ticket, predict the median resolution time of its category" - which is a strong baseline because category alone explains a lot of variance.

In [ ]:
# compute medians on training data, apply to test
medians = (
    train.groupBy('label_canonical')
    .agg(F.expr('percentile_approx(resolution_hours, 0.5)').alias('median_hours'))
)
global_median = float(
    train.approxQuantile('resolution_hours', [0.5], 0.01)[0]
)
print(f'global median resolution: {global_median:.1f} hours ({global_median/24:.1f} days)')

baseline_preds = (
    test.join(medians, on='label_canonical', how='left')
    .fillna({'median_hours': global_median})
    .select('resolution_hours', F.col('median_hours').alias('predicted_hours'))
    .toPandas()
)

y_pred_base = baseline_preds['predicted_hours'].values
y_true_base = baseline_preds['resolution_hours'].values
mae_base = float(np.mean(np.abs(y_true_base - y_pred_base)))
print(f'baseline MAE: {mae_base:.2f} hours ({mae_base/24:.2f} days)')

improvement_pct = 100.0 * (mae_base - mae) / mae_base
print(f'\nlinear regression: {mae:.2f} hours')
print(f'baseline:          {mae_base:.2f} hours')
print(f'improvement:       {improvement_pct:.1f}%')
if improvement_pct >= 10:
    print('PASSED (>=10% target)')
else:
    print('below 10% target - try adding label_canonical as a feature')

## Cell 8 - Per-category MAE (where does our model help vs hurt?)

Useful for the demo: "the model adds the most value on these categories...".

In [ ]:
import pandas as pd

# attach predicted + baseline back to the test df
test_with_label = test.select('unique_key', 'label_canonical', 'resolution_hours').toPandas()
preds_pdf = preds_pdf.reset_index(drop=True)
test_with_label = test_with_label.reset_index(drop=True)
test_with_label['predicted_hours'] = np.maximum(preds_pdf['predicted_hours'].values, 0.01)

median_map = train.groupBy('label_canonical').agg(
    F.expr('percentile_approx(resolution_hours, 0.5)').alias('median_hours')
).toPandas().set_index('label_canonical')['median_hours'].to_dict()
test_with_label['baseline_pred'] = test_with_label['label_canonical'].map(median_map).fillna(global_median)

test_with_label['err_model'] = (test_with_label['resolution_hours'] - test_with_label['predicted_hours']).abs()
test_with_label['err_baseline'] = (test_with_label['resolution_hours'] - test_with_label['baseline_pred']).abs()

by_cat = test_with_label.groupby('label_canonical').agg(
    support=('resolution_hours', 'size'),
    actual_median_hours=('resolution_hours', 'median'),
    mae_model=('err_model', 'mean'),
    mae_baseline=('err_baseline', 'mean'),
).round(2).sort_values('support', ascending=False)
by_cat['improvement_pct'] = (100 * (by_cat['mae_baseline'] - by_cat['mae_model']) / by_cat['mae_baseline']).round(1)
print('per-category mae (model vs baseline):')
print(by_cat.to_string())

by_cat.to_json('/content/project/dashboard/assets/regress_by_category.json', orient='index', indent=2)
print('\nsaved regress_by_category.json')

## Cell 9 - Save model + portable export

Two artifacts: full Spark model on Drive, slim portable artifact in repo for the deployed dashboard.

In [ ]:
# full model on drive
model_path = '/content/drive/MyDrive/cs6513/models/regressor_lr'
lr_model.write().overwrite().save(model_path)
print(f'full PipelineModel saved to {model_path}')

# portable artifact - LR coefficients + categorical label maps
lr_stage = lr_model.stages[-1]  # LinearRegressionModel
agency_labels = lr_model.stages[0].labels
borough_labels = lr_model.stages[1].labels

portable_path = '/content/project/models/portable/regressor.npz'
os.makedirs(os.path.dirname(portable_path), exist_ok=True)
np.savez_compressed(
    portable_path,
    coefs=lr_stage.coefficients.toArray(),
    intercept=float(lr_stage.intercept),
    agency_labels=np.array(agency_labels, dtype=object),
    borough_labels=np.array(borough_labels, dtype=object),
    text_features=4096,
    median_map=np.array(list(median_map.items()), dtype=object),
    global_median=global_median,
)
size_mb = os.path.getsize(portable_path) / 1024 / 1024
print(f'portable regressor saved to {portable_path} ({size_mb:.2f} MB)')

## Cell 10 - Save summary JSON for the dashboard

In [ ]:
import json, datetime

summary = {
    'phase': 4,
    'trained_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'n_train': int(n_train),
    'n_test': int(n_test),
    'model': 'tf-idf + agency_oh + borough_oh + temporal + linear regression on log1p(hours)',
    'metrics_hours_space': {
        'mae': float(mae),
        'rmse': float(rmse),
        'r2': float(r2),
    },
    'baseline_median_per_category': {
        'mae': float(mae_base),
    },
    'improvement_pct': float(improvement_pct),
    'training_time_sec': float(t_fit),
}
with open('/content/project/dashboard/assets/regressor_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('saved regressor_summary.json:')
print(json.dumps(summary, indent=2, default=str))

## Cell 11 - v2 with `label_canonical` as a feature

The v1 model above missed the 10% target (3.8% lift). The v1 had no clean way to *recognize the category* from short descriptors. v2 adds `label_canonical` as a feature directly. In the deployed dashboard the classifier from Phase 3 predicts the category first, then v2 uses that prediction.

This is the realistic operational pipeline: classifier -> regressor.

In [ ]:
# v2 model: add label_canonical (category) as a categorical feature.

cat_idx = StringIndexer(inputCol='label_canonical', outputCol='cat_idx', handleInvalid='keep')
cat_oh = OneHotEncoder(inputCol='cat_idx', outputCol='cat_vec')

assembler_v2 = VectorAssembler(
    inputCols=['text_vec', 'agency_vec', 'borough_vec', 'cat_vec', 'hour_of_day', 'day_of_week'],
    outputCol='features_v2',
)

# fresh estimator pointing at features_v2 to avoid colliding with v1 column names
lr_v2_alg = LinearRegression(
    labelCol='log_resolution_hours',
    featuresCol='features_v2',
    regParam=0.1,
    elasticNetParam=0.0,
    maxIter=50,
)

pipeline_v2 = Pipeline(stages=[
    agency_idx, borough_idx, cat_idx,
    agency_oh, borough_oh, cat_oh,
    htf, idf, assembler_v2, lr_v2_alg,
])

t0 = time.time()
lr_model_v2 = pipeline_v2.fit(train)
t_fit_v2 = time.time() - t0
print(f'v2 fit in {t_fit_v2:.1f} sec')

# evaluate v2 in original hours space
preds_v2 = lr_model_v2.transform(test).select(
    'resolution_hours',
    F.expm1('prediction').alias('predicted_hours'),
).toPandas()
y_true_v2 = preds_v2['resolution_hours'].values
y_pred_v2 = np.maximum(preds_v2['predicted_hours'].values, 0.01)

mae_v2 = float(np.mean(np.abs(y_true_v2 - y_pred_v2)))
rmse_v2 = float(np.sqrt(np.mean((y_true_v2 - y_pred_v2) ** 2)))
ss_res_v2 = float(np.sum((y_true_v2 - y_pred_v2) ** 2))
ss_tot_v2 = float(np.sum((y_true_v2 - y_true_v2.mean()) ** 2))
r2_v2 = 1.0 - ss_res_v2 / ss_tot_v2 if ss_tot_v2 > 0 else 0.0

improvement_v2 = 100.0 * (mae_base - mae_v2) / mae_base

print()
print(f'v1 (text + agency + borough + temporal):       MAE = {mae:7.2f}h, lift = {100*(mae_base-mae)/mae_base:5.1f}%')
print(f'v2 (text + agency + borough + CAT + temporal): MAE = {mae_v2:7.2f}h, lift = {improvement_v2:5.1f}%')
print(f'baseline (median per category):                MAE = {mae_base:7.2f}h')

if improvement_v2 >= 10:
    print('\nv2 PASSED (>=10% target)')
else:
    print(f'\nv2 still below target by {10 - improvement_v2:.1f} pts - try lowering regParam or adding interactions')

In [ ]:
# save v2 portable + spark model + update summary

lr_stage_v2 = lr_model_v2.stages[-1]
agency_labels = lr_model_v2.stages[0].labels
borough_labels = lr_model_v2.stages[1].labels
cat_labels = lr_model_v2.stages[2].labels

portable_path = '/content/project/models/portable/regressor.npz'
np.savez_compressed(
    portable_path,
    coefs=lr_stage_v2.coefficients.toArray(),
    intercept=float(lr_stage_v2.intercept),
    agency_labels=np.array(agency_labels, dtype=object),
    borough_labels=np.array(borough_labels, dtype=object),
    cat_labels=np.array(cat_labels, dtype=object),
    text_features=4096,
    median_map=np.array(list(median_map.items()), dtype=object),
    global_median=global_median,
    version='v2_with_category',
)
size_mb = os.path.getsize(portable_path) / 1024 / 1024
print(f'v2 portable saved ({size_mb:.2f} MB)')

# spark model on drive (replace v1)
v2_model_path = '/content/drive/MyDrive/cs6513/models/regressor_lr_v2'
lr_model_v2.write().overwrite().save(v2_model_path)
print(f'v2 spark model saved to {v2_model_path}')

# update summary json with v2 metrics + retain v1 for comparison
summary['model'] = 'v2: tf-idf + agency_oh + borough_oh + cat_oh + temporal + linear regression on log1p(hours)'
summary['metrics_hours_space'] = {'mae': float(mae_v2), 'rmse': float(rmse_v2), 'r2': float(r2_v2)}
summary['improvement_pct'] = float(improvement_v2)
summary['training_time_sec'] = float(t_fit_v2)
summary['v1_metrics_hours_space'] = {'mae': float(mae), 'rmse': float(rmse), 'r2': float(r2)}
summary['v1_improvement_pct'] = float(improvement_pct)

with open('/content/project/dashboard/assets/regressor_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('updated regressor_summary.json with v2 metrics + v1 archive')

## Phase 4 - Done when

- Cell 6 prints MAE in hours.
- Cell 7 shows model MAE >= 10% better than the median-per-category baseline.
- Cell 8 shows per-category breakdown saved to `regress_by_category.json`.
- Cell 9 reports portable artifact size under 5 MB.
- Cell 10 saves `regressor_summary.json`.

Save the notebook back to GitHub and drop the new PRINT pdf in the project directory. Then we move to Phase 5 (Word2Vec + clustering).